# BCUL importer Debug for new batches

### Imports

In [1]:
# Automatically reloads modules when you make changes (useful during development)
%load_ext autoreload
%autoreload 2

In [2]:
import logging
import os
import json
import string
from collections import namedtuple
import sys
import tqdm

from dask import bag as db


In [3]:
# add the parent directory to the path
sys.path.append("/rcp-scratch/iccluster040_scratch/students/banuls/impresso-essentials")

In [4]:
from text_preparation.importers.detect import _apply_datefilter
from text_preparation.importers.bcul.helpers import parse_date, find_mit_file
from text_preparation.importers.bcul.classes import BculNewspaperIssue


logger = logging.getLogger(__name__)

BculIssueDir = namedtuple(
    "IssueDirectory", ["provider", "alias", "date", "edition", "path", "mit_file_type"]
)

In [5]:
BASE_DIR = "/mnt/project_impresso/original/BCUL"

In [6]:
OLD_ALIASES_FILEPATH = '/rcp-scratch/iccluster040_scratch/students/banuls/impresso-text-acquisition/text_preparation/data/sample_data/BCUL/bcul_aliases.json'
ALIASES_FILEPATH = '/rcp-scratch/iccluster040_scratch/students/banuls/impresso-text-acquisition/text_preparation/data/sample_data/BCUL/bcul_aliases3_4.json'


## Detect


In [7]:
def old_dir2issue(path: str, journal_info: dict[str, str]) -> BculIssueDir | None:
    """Create a `BculIssueDir` object from a directory.

    Note:
        This function is called internally by `detect_issues`

    Args:
        path (str): The path of the issue.
        access_rights (dict): Dictionary for access rights.

    Returns:
        BculIssueDir | None: New `BculIssueDir` object.
    """
    mit_file = find_mit_file(path)
    if mit_file is None:
        logger.error("Could not find MIT file in %s", path)
        return None

    mit_ext = mit_file.split(".")[-1]
    expected_ext = journal_info["file_type"]
    print('mit file ends with:', mit_file, mit_ext, expected_ext)
    # --- handle 'both' case --- 
    if expected_ext == "both":
        if mit_ext not in ('xml', 'json'):
            logger.warning(
                "Found mit file %s has unexpected extension %s, expected 'xml' or 'json'",
                os.path.join(path, mit_file),
                mit_ext,
            )
            # accept either format without changing journal_info
    else: 
        # --- normal case ---
        if not mit_file.endswith(journal_info["file_type"]):
            logger.warning(
                "Found mit file %s does not correspond to mit file type %s",
                os.path.join(path, mit_file),
                expected_ext,
            )
            # override the mit file type if the extension of the file found does not match
            journal_info["file_type"] = mit_ext

    date = parse_date(mit_file)

    # check if multiple issues are at this date:
    day_dir = os.path.dirname(path)
    day_editions = list(os.listdir(day_dir))
    day_editions = [
        str(i)
        for i in os.listdir(day_dir)
        if i != ".DS_Store"
    ]

    if len(day_editions) > 1:
        # if multiple issues exist for a given day, find the correct edition
        logger.info("Multiple issues for %s, finding the edition", day_dir)
        # exclude incorrect issues from the list
        index = sorted(day_editions).index(os.path.basename(path))
        edition = string.ascii_lowercase[index]
    else:
        edition = "a"

    return BculIssueDir(
        provider="BCUL",
        alias=journal_info["alias"],
        date=date,
        edition=edition,
        path=path,
        mit_file_type=mit_ext if expected_ext == "both" else journal_info["file_type"],
    )


In [13]:
def dir2issue(path: str, journal_info: dict[str, str]) -> BculIssueDir | None:
    """Create a `BculIssueDir` object from a directory.

    Note:
        This function is called internally by `detect_issues`

    Args:
        path (str): The path of the issue.
        access_rights (dict): Dictionary for access rights.

    Returns:
        BculIssueDir | None: New `BculIssueDir` object.
    """
    mit_file = find_mit_file(path)
    if mit_file is None:
        logger.error("Could not find MIT file in %s", path)
        return None

    mit_ext = mit_file.split(".")[-1]
    expected_ext = journal_info["mit_file_type"]
    print('mit file ends with:', mit_file, mit_ext, expected_ext)
    # --- handle 'both' case --- 
    if expected_ext == "both":
        if mit_ext not in ('xml', 'json'):
            logger.warning(
                "Found mit file %s has unexpected extension %s, expected 'xml' or 'json'",
                os.path.join(path, mit_file),
                mit_ext,
            )
            # accept either format without changing journal_info
    else: 
        # --- normal case ---
        if not mit_file.endswith(journal_info["mit_file_type"]):
            logger.warning(
                "Found mit file %s does not correspond to mit file type %s",
                os.path.join(path, mit_file),
                expected_ext,
            )
            # override the mit file type if the extension of the file found does not match
            journal_info["mit_file_type"] = mit_ext

    date = parse_date(mit_file)

    # check if multiple issues are at this date:
    day_dir = os.path.dirname(path)
    day_editions = list(os.listdir(day_dir))
    day_editions = [
        str(i)
        for i in os.listdir(day_dir)
        if i != ".DS_Store"
    ]

    if len(day_editions) > 1:
        # if multiple issues exist for a given day, find the correct edition
        logger.info("Multiple issues for %s, finding the edition", day_dir)
        # exclude incorrect issues from the list
        index = sorted(day_editions).index(os.path.basename(path))
        edition = string.ascii_lowercase[index]
    else:
        edition = "a"

    return BculIssueDir(
        provider="BCUL",
        alias=journal_info["alias"],
        date=date,
        edition=edition,
        path=path,
        mit_file_type=mit_ext if expected_ext == "both" else journal_info["mit_file_type"],
    )


## DO NOT RUN IT TAKES FOREVER

In [ ]:
# open and read bcul_alias.json file
with open(ALIASES_FILEPATH, "rb") as f:
    alias_mapping = json.load(f)

dir_path, dirs, files = next(os.walk(BASE_DIR))

journal_dirs = [
    os.path.join(dir_path, _dir)
    for _dir in dirs
    if _dir not in ["OLD", "wrong_BCUL", ".DS_Store"] and _dir in alias_mapping
]
issue_dirs = []
for journal in journal_dirs:
    logger.info("Detecting issues for %s.", journal)
    for dir_path, dirs, files in os.walk(journal):
        title = journal.split("/")[-1]
        # check if we are in the directory of a (valid) issue
        if (
            len(files) > 1
            and "solr" not in dir_path
        ):
            issue_dirs.append(dir2issue(dir_path, alias_mapping[title]))

# return issue_dirs

In [8]:
journal = "/mnt/project_impresso/original/BCUL/Domaine_Public"

In [88]:
issue_dirs = []
nb_issues = 0
for dir_path, dirs, files in os.walk(journal):
    title = journal.split("/")[-1]
    if (
            len(files) > 1
            and "solr" not in dir_path
        ):
            nb_issues += 1
            issue_dirs.append(dir2issue(dir_path, alias_mapping[title]))


    

## bcul.classes.py

In [14]:
# open and read bcul_alias.json file
with open(OLD_ALIASES_FILEPATH, "rb") as f:
    old_alias_mapping = json.load(f)

In [15]:
# open and read bcul_alias.json file
with open(ALIASES_FILEPATH, "rb") as f:
    alias_mapping = json.load(f)

In [50]:
MEdir = old_dir2issue("/mnt/project_impresso/original/BCUL/Le_Grelot/1845/09/01/127522", old_alias_mapping["Le_Grelot"])
CONFdir = dir2issue("/mnt/project_impresso/original/BCUL/Confiance/1950/00/00/394437", alias_mapping["Confiance"])
DPdir = dir2issue("/mnt/project_impresso/original/BCUL/Domaine_Public/1987/10/15/169220", alias_mapping["Domaine_Public"])

mit file ends with: /mnt/project_impresso/original/BCUL/Le_Grelot/1845/09/01/127522/Grelot_0012_1845_09_01_0001_mit.xml xml xml
mit file ends with: /mnt/project_impresso/original/BCUL/Confiance/1950/00/00/394437/EM_1950_09_00_mit.json json json
mit file ends with: /mnt/project_impresso/original/BCUL/Domaine_Public/1987/10/15/169220/DP_0879_1987_10_15_01_mit.xml xml both


In [51]:
print(DPdir)

IssueDirectory(provider='BCUL', alias='DP', date=datetime.date(1987, 10, 15), edition='a', path='/mnt/project_impresso/original/BCUL/Domaine_Public/1987/10/15/169220', mit_file_type='xml')


In [52]:
issue = BculNewspaperIssue(DPdir)
for p in issue.pages:
    print(p.page_data["id"], p.page_data.get("fw"), p.page_data.get("fh"))


DP-1987-10-15-a-p0001 2256 3015
DP-1987-10-15-a-p0002 2256 3015
DP-1987-10-15-a-p0003 2256 3015
DP-1987-10-15-a-p0004 2256 3015
DP-1987-10-15-a-p0005 2256 3015
DP-1987-10-15-a-p0006 2256 3015
DP-1987-10-15-a-p0007 2256 3015
DP-1987-10-15-a-p0008 2256 3015


In [25]:
issue.pages[0].page_data


{'id': 'Grelot-1845-09-01-a-p0001',
 'cdt': '2025-10-28 17:26:20',
 'ts': '2025-10-28T16:26:20Z',
 'st': 'newspaper',
 'sm': 'print',
 'r': [],
 'iiif_img_base_uri': 'https://www.scriptorium.ch/api/iiif-img/v3/523539',
 'fw': 2716,
 'fh': 4611}

In [14]:
import requests, json
url = "https://scriptorium.bcu-lausanne.ch/api/iiif/168346/manifest"
response = requests.get(url, verify=False)
print(response.status_code)

200


In [34]:
file_path = '/mnt/project_impresso/original/BCUL/Confiance/1950/00/00/394437/6355589_exif.json'
with open(file_path, 'r', encoding='utf-8') as jf:
    exif_list = json.load(jf)


In [ ]:
exif_data = exif_list[0]
jpeg_info = exif_data.get("Jpeg2000", {})

In [44]:
w = jpeg_info.get('ImageWidth')
h = jpeg_info.get('ImageHeight')

### Inspect content items of one issue

In [53]:
for ci in issue.content_items[:5]:
    print("CI ID:", ci["m"]["id"])
    print("Type:", ci["m"]["tp"])
    print("Pages:", ci["m"]["pp"])
    print("Legacy info:", ci.get("l"))
    print("IIIF link:", ci["m"].get("iiif_link"))
    print("-" * 50)

CI ID: DP-1987-10-15-a-i0001
Type: page
Pages: [1]
Legacy info: {'id': 'DP-1987-10-15-a-p0001', 'parts': [{'comp_role': 'page', 'comp_id': 'DP-1987-10-15-a-p0001', 'comp_fileid': 'DP_0879_1987_10_15_01_page_1.xml', 'comp_page_no': 1}], 'source': 'DP_0879_1987_10_15_01_page_1.xml'}
IIIF link: None
--------------------------------------------------
CI ID: DP-1987-10-15-a-i0002
Type: page
Pages: [2]
Legacy info: {'id': 'DP-1987-10-15-a-p0002', 'parts': [{'comp_role': 'page', 'comp_id': 'DP-1987-10-15-a-p0002', 'comp_fileid': 'DP_0879_1987_10_15_01_page_2.xml', 'comp_page_no': 2}], 'source': 'DP_0879_1987_10_15_01_page_2.xml'}
IIIF link: None
--------------------------------------------------
CI ID: DP-1987-10-15-a-i0003
Type: page
Pages: [3]
Legacy info: {'id': 'DP-1987-10-15-a-p0003', 'parts': [{'comp_role': 'page', 'comp_id': 'DP-1987-10-15-a-p0003', 'comp_fileid': 'DP_0879_1987_10_15_01_page_3.xml', 'comp_page_no': 3}], 'source': 'DP_0879_1987_10_15_01_page_3.xml'}
IIIF link: None
----